In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [2]:
%%writefile requirements-dev.txt
pytest==8.4.2
huggingface_hub

Writing requirements-dev.txt


In [3]:
!pip install -r requirements-dev.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pytest]

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [5]:
!mkdir -p tests

In [6]:
%%writefile tests/test_pages.py
"""Day 32：自動測試每一頁。執行：python -m pytest -q"""
import json
from pathlib import Path

import pytest
from streamlit.testing.v1 import AppTest

ROOT = Path(__file__).resolve().parent.parent
PAGES = ["pages/home.py", "pages/academy.py", "pages/watch.py", "pages/predict.py"]

def open_page(page):
    at = AppTest.from_file(str(ROOT / "app.py"), default_timeout=120).run()
    if page != "pages/home.py":
        at.switch_page(page).run()
    return at

@pytest.mark.parametrize("page", PAGES)
def test_page_runs_without_error(page):
    at = open_page(page)
    assert not at.exception, [e.value for e in at.exception]

Writing tests/test_pages.py


In [7]:
!python -m pytest -q

....                                                                     [100%]
4 passed in 1.30s


In [8]:
%%writefile -a tests/test_pages.py
def test_every_stat_card():
    at = open_page("pages/academy.py")
    stats = json.loads((ROOT / "content" / "stats.json").read_text(encoding="utf-8"))
    for s in stats:
        at.radio[0].set_value(s["key"]).run()
        assert not at.exception, s["key"]

def test_lineup_needs_nine_players():
    at = open_page("pages/predict.py")
    ms = at.multiselect[0]
    ms.unselect(ms.value[0]).run()
    assert any("9 位打者" in i.value for i in at.info)

def test_series_probabilities_add_up():
    import pandas as pd
    import sys
    sys.path.insert(0, str(ROOT))
    from src import config, model
    teams = pd.read_csv(ROOT / "data/clean/teams.csv").set_index("team")
    a, b = config.TS_TEAMS
    p, results, _ = model.simulate_series(teams.loc[a, "pythag"], teams.loc[b, "pythag"],
                                          config.TS_HOME_PATTERN)
    assert 0 < p < 1
    assert sum(results.values()) == 10_000

Appending to tests/test_pages.py


In [9]:
%%writefile scripts/check_before_deploy.py
"""上線前檢查：跑頁面測試和實驗 1、2，結果寫進 logs/stability_log.csv；沒通過就回傳錯誤碼，不准部署。"""
import datetime as dt
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent

tests_ok = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT).returncode == 0
subprocess.run([sys.executable, "scripts/run_experiments.py"], cwd=ROOT, check=True)
summary = json.loads((ROOT / "data/clean/stability_summary.json").read_text(encoding="utf-8"))

row = {"時間": dt.datetime.now().strftime("%Y-%m-%d %H:%M"), "頁面測試": tests_ok,
       "實驗1": summary["exp1"]["通過"], "實驗2": summary["exp2"]["通過"],
       "實驗3區間寬度": round(summary["exp3"]["區間寬度"], 3), "實驗4最大差距": round(summary["exp4"]["最大差距"], 3)}
log = ROOT / "logs" / "stability_log.csv"
log.parent.mkdir(exist_ok=True)
new = not log.exists()
with log.open("a", encoding="utf-8") as f:
    if new:
        f.write(",".join(row) + "\n")
    f.write(",".join(str(v) for v in row.values()) + "\n")

ok = tests_ok and row["實驗1"] and row["實驗2"]
print("檢查通過，可以部署" if ok else "檢查沒通過，先不要部署")
sys.exit(0 if ok else 1)

Writing scripts/check_before_deploy.py


In [10]:
!python scripts/check_before_deploy.py
!cat logs/stability_log.csv

.......                                                                  [100%]
7 passed in 1.69s
2026-11-05 20:44:43.873 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:44:43.873 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:44:43.873 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:44:43.873 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:44:43.873 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
exp1 {"平均": 0.67072, "標準差": 0.004571380233734595, "通過": true}
exp2 {"最大變動": 0.053099999999999925, "先發局數": {"5": 0.5057455641084038, "6": 0.4998950628748998, "7": 0.49373908457796556}, "看好的球隊不變": true, "通過": true}
exp3 {"奪冠機率95%區間": [0.345375, 0.90375], "區間寬度": 0.5583750000000001, "

In [11]:
!python scripts/update_data.py   # 需要更新資料時才跑
!python scripts/deploy.py        # 自動先檢查，通過才上傳

半季戰績來源：中職官網；交叉比對全部一致
球隊 6、打者 183、投手 181；wOBA 倍數 = 1.001、FIP 常數 = 3.02
逐打席 27268 筆；聯盟 wOBA 0.319
Traceback (most recent call last):
  File "/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/scripts/deploy.py", line 9, in <module>
    if subprocess.run([sys.executable, "scripts/check_before_deploy.py"], cwd=ROOT).returncode != 0:
       ^^^^^^^^^^
NameError: name 'subprocess' is not defined. Did you forget to import 'subprocess'?
